# Load triples to HANA Cloud KG Engine

In [4]:
#Credentials
address = '291933b3-a304-499f-8649-14ec7ce3ce9a.hana.prod-ap10.hanacloud.ondemand.com'
port = '443'
user = 'DBUSER'
password = 'Testing123'

In [5]:
from hdbcli import dbapi

# connect to database using username/password
conn = dbapi.connect(user=user, password=password, address=address, port=port)
cursor = conn.cursor()

In [ ]:
#Populate a new RDF in HANA Cloud with the ttl file content. We put the ontology into a specific graph.
ttl_filename = "/ontology-personnel-fitness-training.ttl"
graph_name = "http://www.semanticweb.org/ontologies/2025/service-member-ontology/"

In [21]:
# Drop existing one if needed
query = """drop graph <{graph_name}>""". format(graph_name="advisory-rdf")
# print(query)

In [22]:
# Uncomment to drop the ontology
# resp = conn.cursor().callproc('SPARQL_EXECUTE', (query, '', '?', None) )

In [23]:
with open(ttl_filename, 'r') as ttlfp:
    request_hdrs = ''
    request_hdrs += 'rqx-load-protocol: true' + '\r\n'            # required header for upload protocol
    request_hdrs += 'rqx-load-filename: ' + ttl_filename + '\r\n' # optional header
    request_hdrs += 'rqx-load-graphname: ' + graph_name + '\r\n'   # optional header to specify name of the graph, 
                                                                  #if not provided RDF data will be loaded to internal-default-graph
    conn.cursor().callproc('SPARQL_EXECUTE', (ttlfp.read(), request_hdrs, '?', None))

In [24]:
query = """
    SELECT * 
    FROM SPARQL_TABLE('
        PREFIX : <http://www.semanticweb.org/ontologies/2025/service-member-ontology/>
        SELECT ?s 
        FROM <http://www.semanticweb.org/ontologies/2025/service-member-ontology/> 
        WHERE { 
            ?s a :ServiceMember
            }
        '
        )"""

In [25]:
#There is just one individual
cursor.execute(query)
for row in cursor:
    print(row)

('http://www.semanticweb.org/ontologies/2025/service-member-ontology/JohnSmith',)
('http://www.semanticweb.org/ontologies/2025/service-member-ontology/JaneDoe',)
('http://www.semanticweb.org/ontologies/2025/service-member-ontology/MikeJohnson',)
('http://www.semanticweb.org/ontologies/2025/service-member-ontology/RobertBrown',)


In [ ]:
# Function to load more ttl files
def load_file(ttl_filename, graph_name):
    try:
        with open(ttl_filename, 'r') as ttlfp:
            request_hdrs = ''
            request_hdrs += 'rqx-load-protocol: true' + '\r\n'            # required header for upload protocol
            request_hdrs += 'rqx-load-filename: ' + ttl_filename + '\r\n' # optional header
            request_hdrs += 'rqx-load-graphname: ' + graph_name + '\r\n'  # optional header to specify name of the graph, 
                                                                          # if not provided RDF data will be loaded 
                                                                          # to internal-default-graph
            conn.cursor().callproc('SPARQL_EXECUTE', (ttlfp.read(), request_hdrs, '?', None))
        return 0
    except:
        print("An exception occurred")
        return 1

In [33]:
# Count the number of triples in the rdf
query = """
    SELECT (COUNT(*) as ?Triples) 
    WHERE 
      { GRAPH <http://www.semanticweb.org/ontologies/2025/service-member-ontology/> 
          { ?s ?p ?o } 
      }
"""
resp = conn.cursor().callproc('SPARQL_EXECUTE', (query, 'Accept: application/sparql-results+csv', '?', None) )
print(resp[2]) 

Triples
329



In [34]:
# Obtain a list of classes in the ontology
query = """
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix : <http://www.semanticweb.org/ontologies/2025/service-member-ontology/> 
    prefix owl: <http://www.w3.org/2002/07/owl#> 
    
    SELECT DISTINCT ?c
    FROM <http://www.semanticweb.org/ontologies/2025/service-member-ontology/>
    WHERE {
        ?c a owl:Class
    }
"""
resp = conn.cursor().callproc('SPARQL_EXECUTE', (query, 'Accept: application/sparql-results+csv', '?', None) )
print(resp[2]) 

c
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Unit
http://www.semanticweb.org/ontologies/2025/service-member-ontology/ServiceMember
http://www.semanticweb.org/ontologies/2025/service-member-ontology/ObligationRule
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Location
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Deferment
http://www.semanticweb.org/ontologies/2025/service-member-ontology/ExitPermit
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Captain
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Corporal
http://www.semanticweb.org/ontologies/2025/service-member-ontology/FitnessTestBooking
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Sergeant
http://www.semanticweb.org/ontologies/2025/service-member-ontology/Lieutenant
http://www.semanticweb.org/ontologies/2025/service-member-ontology/EligibilityRule
http://www.semanticweb.org/ontologies/2025/service-member-

In [ ]:
# Query all object properties defined in your ontology
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX : <http://www.semanticweb.org/ontologies/2025/service-member-ontology/>

SELECT DISTINCT ?p
WHERE {
    ?p a owl:ObjectProperty .
    FILTER(STRSTARTS(STR(?p), "http://www.semanticweb.org/ontologies/2025/service-member-ontology/"))
}
"""

# Execute SPARQL
resp = conn.cursor().callproc(
    'SPARQL_EXECUTE',
    (query, 'Accept: application/sparql-results+csv', '?', None)
)

# Print results (excluding only the header row)
lines = resp[2].splitlines()
if len(lines) > 1:
    for line in lines[1:]:
        print(line)
else:
    print("No object properties found. Make sure your TTL is loaded into the RDF graph.")


http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasObligation
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasExitPermit
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasTime
http://www.semanticweb.org/ontologies/2025/service-member-ontology/occursAt
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasScope
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasBooking
http://www.semanticweb.org/ontologies/2025/service-member-ontology/servesIn
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasFitnessStatus
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasRank
http://www.semanticweb.org/ontologies/2025/service-member-ontology/appliesTo
http://www.semanticweb.org/ontologies/2025/service-member-ontology/hasDeferment


## Other queries

In [35]:
# Get the onthology schema 
query = """
    CONSTRUCT {?s ?p ?o} FROM <http://www.semanticweb.org/ontologies/2025/service-member-ontology/> 
    WHERE {?s ?p ?o}
    LIMIT 10
"""
resp = conn.cursor().callproc('SPARQL_EXECUTE', (query, 'Accept: application/sparql-results+csv', '?', None) )
print(resp[2])

@prefix xsd: <http://www.w3.org/2001/XMLSchema#>.
<http://xmlns.com/foaf/0.1/mbox> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#DatatypeProperty> .
<http://www.semanticweb.org/ontologies/2025/nsmen-ontology/ExitPermit_2025_01> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.semanticweb.org/ontologies/2025/nsmen-ontology/ExitPermit> .
<http://www.semanticweb.org/ontologies/2025/nsmen-ontology/Fitness_Centre_Khatib> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.semanticweb.org/ontologies/2025/nsmen-ontology/Location> .
<http://www.w3.org/2006/time#hasEnd> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#DatatypeProperty> .
<http://www.w3.org/2006/time#hasBeginning> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#DatatypeProperty> .
<http://www.semanticweb.org/ontologies/2025/nsmen-ontology/IPPTBooking_2025_0001> <http://www.w3.org/1999/02/22-rdf-syntax-ns#typ